# 05 — WBIC Model Comparison

Loads the per-session and per-subject WBIC CSVs produced by `04_wbic_fitting.ipynb`
and generates comparison plots.

**Figures produced**
1. WBIC by model — bar/dot plot per stage (one panel per stage, dots = subjects)
2. WBIC across stages — line plot per model (group mean ± SEM across subjects)
3. Parameter estimates — one subplot per parameter per model across stages

In [ ]:
# ── Cell 2: Imports & path setup ──────────────────────────────────────────

import sys, os

_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    _organized = os.path.dirname(_here)
else:
    _organized = os.path.join(_here, 'organized')
    if not os.path.isdir(_organized):
        _organized = _here
sys.path.insert(0, _organized)
# Also add src/ so that pickle can reconstruct Session objects whose
# class was saved as 'data_import.Session' (not 'src.data_import.Session').
sys.path.insert(0, os.path.join(_organized, 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from config import SUBJECTS, FIGURES_DIR, wbic_result_csv, wbic_subj_csv
from src.wbic import WBIC_MODELS

os.makedirs(FIGURES_DIR, exist_ok=True)
print('Figures will be saved to:', FIGURES_DIR)

In [ ]:
# ── Cell 2: Imports & path setup ──────────────────────────────────────────

import sys, os

_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    _organized = os.path.dirname(_here)
else:
    _organized = os.path.join(_here, 'organized')
    if not os.path.isdir(_organized):
        _organized = _here
sys.path.insert(0, _organized)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from config import SUBJECTS, FIGURES_DIR, wbic_result_csv, wbic_subj_csv
from src.wbic import WBIC_MODELS

os.makedirs(FIGURES_DIR, exist_ok=True)
print('Figures will be saved to:', FIGURES_DIR)

In [ ]:
# ── Cell 3: Load all per-session data ─────────────────────────────────────

dfs = []
for model_key in MODELS_TO_COMPARE:
    for stage in STAGES:
        csv_path = wbic_result_csv(model_key, stage)
        if os.path.exists(csv_path):
            dfs.append(pd.read_csv(csv_path))

if not dfs:
    raise FileNotFoundError(
        'No WBIC CSVs found. Run 04_wbic_fitting.ipynb first.')

df_all = pd.concat(dfs, ignore_index=True)
df_all['model_label'] = df_all['model'].map(MODEL_LABELS)

print(f'Loaded {len(df_all)} session rows.')
print(df_all.groupby(['model', 'stage']).size().unstack(fill_value=0).to_string())

In [ ]:
# ── Cell 4: Load per-subject summaries ────────────────────────────────────

subj_dfs = []
for model_key in MODELS_TO_COMPARE:
    for stage in STAGES:
        subj_path = wbic_subj_csv(model_key, stage)
        if os.path.exists(subj_path):
            tmp = pd.read_csv(subj_path)
            tmp['model'] = model_key
            tmp['stage'] = stage
            subj_dfs.append(tmp)

df_subj = pd.concat(subj_dfs, ignore_index=True) if subj_dfs else pd.DataFrame()
print(f'Loaded subject summaries: {len(df_subj)} rows.')

In [ ]:
# ── Cell 5: Figure 1 — WBIC by model, one panel per stage ─────────────────
#
# For each stage: bar = group mean across subjects, dots = individual subjects.
# Uses per-subject means (WBIC_mean) from the summary CSVs.

if df_subj.empty:
    print('No subject summary data — run Cell 4 first.')
else:
    available_stages = [s for s in STAGES
                        if s in df_subj['stage'].values]
    n_stages = len(available_stages)
    n_cols = min(4, n_stages)
    n_rows = int(np.ceil(n_stages / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(4 * n_cols, 4 * n_rows),
                             sharey=False)
    axes = np.array(axes).flatten()

    x_pos = np.arange(len(MODELS_TO_COMPARE))

    for ax_idx, stage in enumerate(available_stages):
        ax = axes[ax_idx]
        stage_data = df_subj[df_subj['stage'] == stage]

        for m_idx, model_key in enumerate(MODELS_TO_COMPARE):
            model_data = stage_data[stage_data['model'] == model_key]['WBIC_mean']
            if model_data.empty:
                continue

            color = MODEL_COLORS[model_key]
            group_mean = model_data.mean()
            group_sem  = model_data.sem()

            ax.bar(m_idx, group_mean, color=color, alpha=0.6, width=0.6)
            ax.errorbar(m_idx, group_mean, yerr=group_sem,
                        fmt='none', color='black', capsize=4, linewidth=1.5)
            # Individual subject dots
            jitter = np.random.uniform(-0.1, 0.1, size=len(model_data))
            ax.scatter(m_idx + jitter, model_data.values,
                       color=color, edgecolors='black', s=40, zorder=3)

        ax.set_title(f'Stage {stage}', fontsize=11)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(
            [MODEL_LABELS[m] for m in MODELS_TO_COMPARE],
            rotation=30, ha='right', fontsize=8)
        ax.set_ylabel('WBIC', fontsize=9)
        ax.spines[['top', 'right']].set_visible(False)

    # Hide unused axes
    for ax in axes[len(available_stages):]:
        ax.set_visible(False)

    fig.suptitle('WBIC by Model (lower = better fit)', fontsize=13, y=1.01)
    fig.tight_layout()
    save_path = os.path.join(FIGURES_DIR, 'wbic_by_model_per_stage.pdf')
    fig.savefig(save_path, bbox_inches='tight')
    print(f'Saved: {save_path}')
    plt.show()

In [ ]:
# ── Cell 6: Figure 2 — WBIC across stages, one line per model ─────────────
#
# X-axis: stage (4.2 → 4.8), Y-axis: WBIC.
# Line = group mean across subjects; shaded band = ±1 SEM.

if df_subj.empty:
    print('No subject summary data — run Cell 4 first.')
else:
    fig, ax = plt.subplots(figsize=(8, 5))

    for model_key in MODELS_TO_COMPARE:
        model_data = df_subj[df_subj['model'] == model_key]
        if model_data.empty:
            continue

        stage_means = []
        stage_sems  = []
        stage_xs    = []

        for stage in STAGES:
            s_data = model_data[model_data['stage'] == stage]['WBIC_mean']
            if s_data.empty:
                continue
            stage_xs.append(stage)
            stage_means.append(s_data.mean())
            stage_sems.append(s_data.sem())

        if not stage_xs:
            continue

        means = np.array(stage_means)
        sems  = np.array(stage_sems)
        color = MODEL_COLORS[model_key]
        label = MODEL_LABELS[model_key]

        ax.plot(stage_xs, means, marker='o', color=color, label=label, linewidth=2)
        ax.fill_between(stage_xs, means - sems, means + sems,
                        color=color, alpha=0.2)

    ax.set_xlabel('Training Stage', fontsize=12)
    ax.set_ylabel('WBIC (group mean ± SEM)', fontsize=12)
    ax.set_title('WBIC across Training Stages (lower = better fit)', fontsize=13)
    ax.legend(fontsize=9, frameon=False)
    ax.spines[['top', 'right']].set_visible(False)

    fig.tight_layout()
    save_path = os.path.join(FIGURES_DIR, 'wbic_across_stages.pdf')
    fig.savefig(save_path, bbox_inches='tight')
    print(f'Saved: {save_path}')
    plt.show()

In [ ]:
# ── Cell 7: Figure 3 — Parameter estimates across stages ──────────────────
#
# For each model: one subplot per parameter.
# X-axis: stage, Y-axis: parameter value.
# Line = group mean across subjects; shaded = ±1 SEM.

for model_key in MODELS_TO_COMPARE:
    param_names = WBIC_MODELS[model_key].param_names
    n_params = len(param_names)
    color = MODEL_COLORS[model_key]

    model_subj = df_subj[df_subj['model'] == model_key]
    if model_subj.empty:
        print(f'No data for {model_key} — skipping.')
        continue

    n_cols = min(3, n_params)
    n_rows = int(np.ceil(n_params / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(4 * n_cols, 3.5 * n_rows),
                             sharey=False)
    axes = np.array(axes).flatten()

    for p_idx, param in enumerate(param_names):
        ax = axes[p_idx]
        col = f'{param}_mean'

        xs, means, sems = [], [], []
        for stage in STAGES:
            s_data = model_subj[model_subj['stage'] == stage]
            if s_data.empty or col not in s_data.columns:
                continue
            vals = s_data[col].dropna()
            if vals.empty:
                continue
            xs.append(stage)
            means.append(vals.mean())
            sems.append(vals.sem())

        if not xs:
            ax.set_visible(False)
            continue

        means = np.array(means)
        sems  = np.array(sems)

        ax.plot(xs, means, marker='o', color=color, linewidth=2)
        ax.fill_between(xs, means - sems, means + sems, color=color, alpha=0.2)
        ax.set_title(param, fontsize=11)
        ax.set_xlabel('Stage', fontsize=9)
        ax.set_ylabel('Posterior mean', fontsize=9)
        ax.spines[['top', 'right']].set_visible(False)

    for ax in axes[n_params:]:
        ax.set_visible(False)

    fig.suptitle(f'{MODEL_LABELS[model_key]} — Parameter estimates across stages',
                 fontsize=12, y=1.01)
    fig.tight_layout()
    save_path = os.path.join(FIGURES_DIR, f'params_{model_key}_across_stages.pdf')
    fig.savefig(save_path, bbox_inches='tight')
    print(f'Saved: {save_path}')
    plt.show()

In [ ]:
# ── Cell 8: Summary table for paper ───────────────────────────────────────
#
# Group mean ± SD of WBIC per model per stage, suitable for copy-pasting.

if df_subj.empty:
    print('No subject summary data available.')
else:
    rows = []
    for model_key in MODELS_TO_COMPARE:
        model_data = df_subj[df_subj['model'] == model_key]
        for stage in STAGES:
            s_data = model_data[model_data['stage'] == stage]['WBIC_mean']
            if s_data.empty:
                continue
            rows.append({
                'Model': MODEL_LABELS[model_key],
                'Stage': stage,
                'N_subjects': len(s_data),
                'WBIC_mean': round(s_data.mean(), 2),
                'WBIC_SD':   round(s_data.std(), 2),
            })

    df_table = pd.DataFrame(rows)
    # Pivot for compact display: rows=model, columns=stage
    pivot = df_table.pivot_table(
        index='Model', columns='Stage',
        values='WBIC_mean', aggfunc='first'
    )
    print('WBIC group means by model × stage:')
    print(pivot.to_string())

    # Save full table
    table_path = os.path.join(FIGURES_DIR, 'wbic_summary_table.csv')
    df_table.to_csv(table_path, index=False)
    print(f'\nFull table saved: {table_path}')